In [ ]:
import shutil
import os
import glob
from IPython.display import FileLink

# Define output names
master_output = 'Final_Master_PhD_Dataset.csv'
queue_output = 'Queue_Length_Data.csv'

# Function to find and copy files
def save_to_output(search_pattern, output_name):
    found_files = glob.glob(f'/kaggle/input/**/{search_pattern}', recursive=True)
    if found_files:
        shutil.copy(found_files[0], f'/kaggle/working/{output_name}')
        print(f"✅ Success: {output_name} saved to Output.")
        return True
    else:
        print(f"❌ Error: Could not find file matching '{search_pattern}' in Input.")
        return False

print("--- Processing Files ---")
# 1. Save Master Dataset
save_to_output('*Final_Master_PhD_Dataset*.csv', master_output)

# 2. Save Queue Length Data
save_to_output('*Queue_Length_Data*.csv', queue_output)

# 3. Show Download Links
print("\n--- Download Links ---")
if os.path.exists(master_output):
    display(FileLink(master_output))
if os.path.exists(queue_output):
    display(FileLink(queue_output))

# List all files in working directory to verify
print("\nFiles now in your Output Directory:", os.listdir('/kaggle/working/'))

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D

# 1. Automatic File Detection
# This part finds any CSV file in your working directory to avoid FileNotFoundError
csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
target_file = None
for f in csv_files:
    if "Final" in f or "Master" in f or "PhD" in f:
        target_file = f
        break

if not target_file:
    # Fallback to the known name if auto-detect fails
    target_file = 'Final_Master_PhD_Dataset (1).csv'

print(f"--- Loading File: {target_file} ---")
df = pd.read_csv(target_file)

# 2. Flexible Data Filtering (Fuzzy Matching)
# This looks for any row containing 'Travel Time' regardless of case or extra spaces
df.columns = df.columns.str.strip()
data = df[df['Metric'].str.contains('Travel Time', case=False, na=False)].copy()

print(f"Successfully found {len(data)} rows for Travel Time.")

if len(data) == 0:
    print("Available Metrics in your file are:", df['Metric'].unique())
    raise ValueError("Still could not find 'Travel Time' rows. Please check the 'Metric' column values.")

# 3. Feature Engineering
# Extract numbers from taper (e.g. '300m' -> 300)
data['Taper_Numeric'] = data['Condition_Taper'].str.extract(r'(\d+)').fillna(0).astype(float)
# Encode Vehicle categories
le = LabelEncoder()
data['Vehicle_Encoded'] = le.fit_transform(data['Vehicle_Type_Cleaned'].astype(str))
# Ensure Target is numeric
data['Value'] = pd.to_numeric(data['Value'], errors='coerce')
data = data.dropna(subset=['Value'])

X = data[['Taper_Numeric', 'Vehicle_Encoded']]
y = data['Value']

# 4. Machine Learning Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Algorithm 1: k-NN ---
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict(X_test_scaled)

# --- Data Preparation for Deep Learning ---
X_train_dl = np.reshape(X_train_scaled, (X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_dl = np.reshape(X_test_scaled, (X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

# --- Algorithm 2: LSTM ---
lstm = Sequential([
    LSTM(64, activation='relu', input_shape=(1, 2)),
    Dense(1)
])
lstm.compile(optimizer='adam', loss='mae')
lstm.fit(X_train_dl, y_train, epochs=30, verbose=0)
lstm_pred = lstm.predict(X_test_dl).flatten()

# --- Algorithm 3: Transformer (Multi-Head Attention) ---
inputs = Input(shape=(1, 2))
attn_output = MultiHeadAttention(num_heads=2, key_dim=2)(inputs, inputs)
x = LayerNormalization(epsilon=1e-6)(attn_output + inputs)
x = GlobalAveragePooling1D()(x)
outputs = Dense(1)(x)
transformer_model = Model(inputs, outputs)
transformer_model.compile(optimizer='adam', loss='mae')
transformer_model.fit(X_train_dl, y_train, epochs=30, verbose=0)
trans_pred = transformer_model.predict(X_test_dl).flatten()

# --- 5. VISUALIZATION (Comparison Graph) ---
plt.figure(figsize=(12, 6))
plt.plot(y_test.values[:50], label='Actual Values', color='black', linewidth=1.5, marker='o')
plt.plot(trans_pred[:50], label='Transformer (Best)', color='red', linewidth=2)
plt.plot(lstm_pred[:50], label='LSTM', linestyle='--')
plt.plot(knn_pred[:50], label='k-NN', linestyle=':')
plt.title('Performance Comparison for Travel Time Prediction')
plt.ylabel('Seconds')
plt.xlabel('Sample Index')
plt.legend()
plt.savefig('Final_Comparison_Results.png')
plt.show()

# 6. Performance Summary Table
performance_table = pd.DataFrame({
    'Model': ['k-NN', 'LSTM', 'Transformer'],
    'MAE (Error)': [mean_absolute_error(y_test, knn_pred), mean_absolute_error(y_test, lstm_pred), mean_absolute_error(y_test, trans_pred)],
    'R2 Score (Accuracy)': [r2_score(y_test, knn_pred), r2_score(y_test, lstm_pred), r2_score(y_test, trans_pred)]
})
print("\n--- Final Model Comparison ---")
print(performance_table)

In [ ]:
import pandas as pd

# Inspect the messy queue length file
q_file = "/kaggle/input/workzone-dataset/MOE/Queue_Length_Data.csv"
df_raw = pd.read_csv(q_file, header=None)
print("Raw Queue Data Head:")
print(df_raw.head(20))

# Try to see where the taper mentions are
mask = df_raw.apply(lambda row: row.astype(str).str.contains('meter', case=False).any(), axis=1)
print("\nRows with 'meter' mentions:")
print(df_raw[mask])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D

# 1. Load Datasets
master_df = pd.read_csv('/kaggle/input/workzone-dataset/MOE/Final_Master_PhD_Dataset (1).csv')
queue_raw = pd.read_csv('/kaggle/input/workzone-dataset/MOE/Queue_Length_Data.csv', header=None)

# 2. Cleaning Queue Data (Row-by-Row Logic)
cleaned_q_list = []
current_taper = 0

for idx, row in queue_raw.iterrows():
    val_3 = str(row[3]).lower()
    # Heading se Taper nikaalna
    if 'meter' in val_3:
        numbers = re.findall(r'\d+', val_3)
        if numbers:
            current_taper = int(numbers[0])
        continue
    
    # Numeric data check (Queue length aur No of vehicles)
    try:
        q_len = float(row[0])
        v_cnt = float(row[1])
        if not np.isnan(q_len):
            cleaned_q_list.append([current_taper, q_len, v_cnt])
    except:
        continue

queue_df = pd.DataFrame(cleaned_q_list, columns=['Taper_Numeric', 'Queue_Length', 'Vehicle_Count'])
queue_features = queue_df.groupby('Taper_Numeric').mean().reset_index()

# 3. Merge with Master Dataset
master_df['Taper_Numeric'] = master_df['Condition_Taper'].str.extract(r'(\d+)').fillna(0).astype(int)
master_tt = master_df[master_df['Metric'].str.contains('Travel Time', case=False, na=False)].copy()

final_df = pd.merge(master_tt, queue_features, on='Taper_Numeric', how='left')
final_df = final_df.dropna(subset=['Value', 'Queue_Length'])

# 4. Feature Engineering & Scaling
le = LabelEncoder()
final_df['Vehicle_Enc'] = le.fit_transform(final_df['Vehicle_Type_Cleaned'].astype(str))

X = final_df[['Taper_Numeric', 'Vehicle_Enc', 'Queue_Length', 'Vehicle_Count']]
y = final_df['Value']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Reshape for Transformer
X_train_tr = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_tr = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

# 5. Transformer Model
def build_transformer():
    inputs = Input(shape=(1, 4))
    attn = MultiHeadAttention(num_heads=4, key_dim=4)(inputs, inputs)
    x = LayerNormalization()(attn + inputs)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    outputs = Dense(1)(x)
    return Model(inputs, outputs)

model = build_transformer()
model.compile(optimizer='adam', loss='mse')
model.fit(X_train_tr, y_train, epochs=100, verbose=0)

# 6. Final Results
pred = model.predict(X_test_tr).flatten()
print(f"\n✅ PhD Analysis Success!")
print(f"MAE: {mean_absolute_error(y_test, pred):.2f}")
print(f"R2 Score: {r2_score(y_test, pred):.4f}")

# Plotting

plt.figure(figsize=(8,5))
plt.scatter(y_test, pred, alpha=0.6, color='darkblue')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.title('Travel Time Prediction (Integrated Queue Data)')
plt.show()